# ML · Encuentro 4 · Tiempo, acciones y evidencia

El tiempo, las decisiones y el cierre

**Tu copia personal:** en Colab elige **Archivo → Guardar una copia en Drive**. Conserva esa copia para los tres temas del encuentro.

Trabaja en CPU. Estas prácticas no requieren contratar GPU, claves de API ni servicios de pago. La primera preparación puede descargar bibliotecas ausentes; las prácticas de detección descargan los pesos del modelo si no están disponibles.

## Tu recorrido

1. [El futuro va después](#scrollTo=tema-ml-series)
2. [Aprender de las consecuencias](#scrollTo=tema-ml-refuerzo)
3. [Explicar el resultado antes de usarlo](#scrollTo=tema-ml-integracion)

Cada tema reúne **idea → caso explicado → preparación → ejecución → lectura y comprobación**. Puedes ejecutar todo el cuaderno o empezar por un tema: cada uno vuelve a preparar sus datos y herramientas. No mezcles las cifras de conjuntos diferentes.

Los datos inventados se identifican como simulaciones. Las fotos de detección son demostraciones y no incluyen una evaluación anotada de sus objetos. Las comprobaciones son voluntarias y sin nota.


## Tema 1 · 5 · Series temporales

**Lo que vas a poder hacer:** Construir un pronóstico para planear compras y comprobarlo respetando el tiempo.

Planear compras exige anticipar ventas usando solo información disponible al decidir.

### Entiende la idea

Una serie temporal es el historial de una cantidad en orden de fecha: por ejemplo, cajas vendidas cada semana. Un pronóstico es una estimación de una cantidad que todavía no se conoce. Para hacerlo hay que decir hasta qué semana tenemos información y para qué semanas futuras necesitamos la respuesta. La primera separación se llama corte; las semanas que queremos anticipar son el horizonte. Comprobar un pronóstico consiste en esperar o reservar las respuestas posteriores y compararlas con lo que se había estimado, sin usarlas para elegir la regla.

### Un caso pequeño antes del programa

**Pronosticar dos semanas antes de verlas**

Una comercializadora decide su planificación al cierre de la semana 4. Las cantidades inventadas son cajas de tomate vendidas por semana; cada caja tiene el mismo formato comercial en el ejercicio. Las semanas 5–6 están reservadas y se revelan después. Repetir dos semanas es una regla elegida para este caso, no una estacionalidad observada del mercado.

| Semana | Ventas (cajas) | Uso |
| --- | --- | --- |
| 1 | 100 | Conocida |
| 2 | 110 | Conocida |
| 3 | 100 | Conocida |
| 4 | 110 | Conocida |
| 5 | 100 | Reservada |
| 6 | 110 | Reservada |

1. Al cerrar la semana 4, la regla «último valor» pronostica 110 y 110 para las semanas 5 y 6.
2. Repetir las dos últimas semanas conocidas pronostica 100 y 110. Ambas reglas usan únicamente historia disponible.
3. Revelamos la reserva: el último valor tiene MAE = (10 + 0)/2 = 5 unidades; repetir ciclo da cero.

**Qué demuestra:** Para las semanas 5–6, repetir el ciclo de esta tabla da 100 y 110 cajas y MAE 0; repetir el último valor da 110 y 110, con MAE 5 cajas. Para convertir un pronóstico en compra aún harían falta existencias, plazos y consecuencias; el ejemplo no los calcula.

### Ahora identifica los datos de esta práctica

Ubica la separación entre 60 semanas conocidas y 24 reservadas. Lee la tabla y las líneas como comparaciones del mismo futuro.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** La referencia de 12 semanas pertenece a esta simulación. No supone que un año tenga 12 semanas ni describe una temporada agrícola real.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.cluster import DBSCAN
rng = np.random.default_rng(42)


In [ ]:
# Generador local: las reglas siempre se comparan sobre la misma serie simulada.
rng_serie = np.random.default_rng(42)
t=np.arange(84)
ventas=100+.6*t+12*np.sin(2*np.pi*t/12)+rng_serie.normal(0,2,len(t))
corte=60;periodo=12
pasado=ventas[:corte];futuro=ventas[corte:]
t_tr=t[:corte,None];t_te=t[corte:,None]
modelo=LinearRegression().fit(t_tr,pasado)
pronosticos={'Último valor':np.repeat(pasado[-1],len(futuro)),'Referencia estacional':np.tile(pasado[-periodo:],2),'Tendencia lineal':modelo.predict(t_te)}
display(pd.DataFrame({'método':list(pronosticos),'MAE_unidades':[mean_absolute_error(futuro,p) for p in pronosticos.values()]}))
plt.figure(figsize=(10,4));plt.plot(t,ventas,label='Ventas observadas',color='black')
for nombre,p in pronosticos.items(): plt.plot(t[corte:],p,label=nombre)
plt.axvline(corte-.5,linestyle='--',color='gray',label='Fin de entrenamiento');plt.xlabel('Semana');plt.ylabel('Unidades vendidas');plt.title('Pasado para entrenar, futuro para evaluar');plt.legend();plt.show()


### Lectura del gráfico 5 · Primero localiza la fecha de decisión

El eje horizontal numera semanas de **0 a 83**: contar desde cero no elimina una semana. La separación discontinua está entre los índices 59 y 60, después de las 60 semanas conocidas. El eje vertical mide unidades vendidas simuladas. La línea negra es lo observado; a la derecha del corte se enseña después para evaluar, no para ajustar.

**Mira una regla cada vez con la leyenda:**

- Último valor: repite la última venta conocida y dibuja una horizontal.
- Referencia estacional: repite dos veces las últimas 12 semanas conocidas. «Estacional» significa aquí ciclo elegido de 12 semanas, no año agrícola.
- Tendencia lineal: prolonga una recta. Puede captar crecimiento, pero no la oscilación del ciclo.

En una semana reservada, la distancia vertical entre una predicción y la línea negra es su error. El **MAE** promedia las 24 distancias sin signo. Lee los valores que imprime la tabla de esta ejecución; corresponden a esta serie y sus 24 semanas reservadas. La recta puede tener menor error aunque visualmente no siga cada oscilación.

**Tip comprobable:** revisa `len(pasado)` y `len(futuro)`: deben ser 60 y 24. Para ver una semana concreta sin cambiar la simulación, compara `futuro[0]` con cada `p[0]` de los pronósticos. Esa es la primera semana reservada, índice 60. Su diferencia no es el MAE total: este promedia las 24.

Si quieres estudiar otro corte, conserva una copia del resultado y adapta también las referencias al nuevo horizonte; no cambies solo 60 en una línea y des por válido que repetir dos ciclos cubre cualquier longitud. Este ejemplo está preparado para 24 semanas y dos ciclos de 12.


### Lee, comprueba y explica

Ubica la separación entre 60 semanas conocidas y 24 reservadas. Lee la tabla y las líneas como comparaciones del mismo futuro.

**Prueba pequeña:** La referencia de 12 semanas pertenece a esta simulación. No supone que un año tenga 12 semanas ni describe una temporada agrícola real.

**Vuelve al caso:** Registra fecha de corte, horizonte y error. Compara las reglas sobre las mismas fechas posteriores al corte.

**Pregunta:** ¿Por qué una buena predicción del pasado podría fallar como pronóstico?

**Respuesta razonada:** Podría haberse construido usando información que llegó después de la fecha de decisión. Un pronóstico válido usa solo historia disponible y se comprueba en un horizonte posterior al corte.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/ml-series)


## Tema 2 · Q-learning: aprender una ruta en una simulación

**Lo que vas a poder hacer:** Construir y probar una secuencia de acciones con recompensas acumuladas.

Una acción afecta pasos posteriores; no basta con evaluar cada movimiento por separado.

### Entiende la idea

Piensa en recoger un pedido: cada movimiento cambia el lugar desde el que podrás seguir. Aprender por refuerzo es aprender qué acciones convienen probando sus consecuencias dentro de un entorno. El agente es quien elige; el entorno es el mundo del ejercicio que responde a esa elección. El estado describe dónde está; la acción es lo que hace; la recompensa es el número que recibe después. Una política es la regla que le dice qué elegir en cada estado. Primero recorremos una ruta a mano. Después la práctica repite intentos para aprender una regla, mediante un método llamado Q-learning.

### Un caso pequeño antes del programa

**Llegar a la meta sin tocar el pozo**

Representamos un recorrido de preparación de pedidos en una bodega mediante un tablero inventado de 5 × 5 casillas. Inicio: fila 0, columna 0; meta: 1,4. Es una representación didáctica, no un plano de una bodega real. Los puntos −1 por paso, +10 al llegar y −10 al caer al pozo son recompensas elegidas para el ejercicio: no dinero ni minutos. Las coordenadas empiezan en cero.

| Paso | Acción | Estado final (fila, columna) | Recompensa |
| --- | --- | --- | --- |
| 1 | Derecha | 0,1 | −1 |
| 2 | Derecha | 0,2 | −1 |
| 3 | Derecha | 0,3 | −1 |
| 4 | Derecha | 0,4 | −1 |
| 5 | Abajo | 1,4: meta | +10 |

1. Los cuatro movimientos iniciales cambian casilla y cuestan una unidad de recompensa cada uno.
2. Entrar en la meta registra +10 y termina el episodio. La suma sin descuento es −1 −1 −1 −1 +10 = 6.
3. Una pared o borde impide avanzar, pero consume un paso y resta 1. Caer al pozo resta 10 y termina.

**Qué demuestra:** La ficha de esta ruta contiene cinco acciones, llegada a la meta y retorno 6 puntos. Una ruta manual permite comprobar las reglas; no demuestra que se haya aprendido una política ni que sea óptima para cualquier bodega.

### Ahora identifica los datos de esta práctica

Distingue los intentos de entrenamiento de la ruta final. Una flecha representa una acción; la recompensa fue fijada por el ejercicio.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** Mover el número de episodios cambia cuánto practica el agente. No cambia por sí solo las recompensas ni convierte el tablero en una operación real.


In [ ]:
from IPython.display import display
# ======================================================================
# Q-LEARNING TABULAR (Colab listo para pegar)
# Caso didáctico: GridWorld 5x5 con paredes, meta (recompensa +10) y pozo (-10)
# ======================================================================
# Objetivo:
# - Aprender una política π(s) con alto retorno esperado descontado (gamma=0.95).
# - Q-learning estima Q[s, a] (valor de acción) iterando:
#   Q[s,a] ← Q[s,a] + α * (r + γ * max_a' Q[s',a'] - Q[s,a])
#
# Conceptos clave:
# - Tabla Q: matriz estados×acciones con valores estimados.
# - Política ε-greedy: con prob. ε explora; con 1-ε explota el mejor Q.
# - Aprendizaje fuera de política (off-policy): usa el max_a' Q[s',a'].
# ======================================================================

import numpy as np
np.random.seed(42)
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------------
# 1) Definición del entorno GridWorld (discreto y tabular)
# --------------------------------------------------------
# Rejilla 5x5. Estados = celdas accesibles.
# Acciones: 0=Arriba, 1=Abajo, 2=Izquierda, 3=Derecha.
# Recompensas:
# - Paso normal: -1 (incentiva trayecto corto)
# - Meta (G): +10 y episodio termina
# - Pozo (X): -10 y episodio termina
# Paredes (#): no se puede entrar; quedarse en sitio penaliza -1
# Bordes: moverse fuera del tablero no está permitido; se ignora el movimiento y se penaliza -1.

H, W = 5, 5

# Mapa:
# S = inicio, G = meta, X = pozo, # = pared, . = libre
grid = np.array([
    list("S...."),
    list(".#..G"),
    list("..#.."),
    list("..X.."),
    list("....."),
])

start_pos = (0, 0)

# Utilidades de entorno
ACTIONS = [(-1,0), (1,0), (0,-1), (0,1)]  # U, D, L, R
N_ACTIONS = 4

def in_bounds(r, c):
    return 0 <= r < H and 0 <= c < W

def is_wall(r, c):
    return grid[r, c] == '#'

def to_state(r, c):
    return r * W + c

def from_state(s):
    return divmod(s, W)

def reset():
    return to_state(*start_pos)

def step(state, action):
    r, c = from_state(state)
    dr, dc = ACTIONS[action]
    nr, nc = r + dr, c + dc

    # Intento de movimiento
    if not in_bounds(nr, nc) or is_wall(nr, nc):
        nr, nc = r, c  # rebote
        reward = -1
        done = False
    else:
        cell = grid[nr, nc]
        if cell == 'G':
            reward = +10
            done = True
        elif cell == 'X':
            reward = -10
            done = True
        else:
            reward = -1
            done = False

    next_state = to_state(nr, nc)
    return next_state, reward, done

# Estados válidos (todas las celdas no pared)
mask_valid = (grid != '#')
N_STATES = H * W

# --------------------------------------------------------
# 2) Hiperparámetros de Q-learning
# --------------------------------------------------------
alpha = 0.1      # tasa de aprendizaje
gamma = 0.95     # factor de descuento
epsilon_start = 1.0
epsilon_end   = 0.05
epsilon_decay_episodes = 800  # episodios para decaer de start a end

episodes = 2000
max_steps = 200  # seguridad por episodio

# --------------------------------------------------------
# 3) Inicialización de la tabla Q
# --------------------------------------------------------
Q = np.zeros((N_STATES, N_ACTIONS), dtype=np.float32)

# Política ε-greedy
def epsilon_by_episode(ep):
    # Decaimiento lineal simple
    t = min(1.0, ep / epsilon_decay_episodes)
    return epsilon_start + t * (epsilon_end - epsilon_start)

def select_action(state, epsilon):
    if np.random.rand() < epsilon:
        return np.random.randint(N_ACTIONS)
    # Explota el mejor Q; desempata al azar
    qs = Q[state]
    max_q = np.max(qs)
    best_actions = np.flatnonzero(qs == max_q)
    return np.random.choice(best_actions)

# --------------------------------------------------------
# 4) Entrenamiento
# --------------------------------------------------------
returns = []

for ep in range(episodes):
    s = reset()
    total_reward = 0.0
    eps = epsilon_by_episode(ep)

    for t in range(max_steps):
        a = select_action(s, eps)
        s_next, r, done = step(s, a)

        # Regla central de Q-learning (off-policy)
        td_target = r + gamma * np.max(Q[s_next]) * (0 if done else 1)
        td_error  = td_target - Q[s, a]
        Q[s, a]  += alpha * td_error

        s = s_next
        total_reward += r
        if done:
            break

    returns.append(total_reward)

print("Entrenamiento completado.")
print("Recompensa media últimas 100 eps:", np.mean(returns[-100:]))

# --------------------------------------------------------
# 5) Visualización de aprendizaje
# --------------------------------------------------------
plt.figure(figsize=(7,3))
plt.plot(returns, linewidth=1)
plt.title("Suma de recompensas sin descuento por episodio exploratorio")
plt.xlabel("Episodio")
plt.ylabel("Retorno sin descuento")
plt.grid(True)
plt.show()

# --------------------------------------------------------
# 6) Extraer política greedy aprendida y mostrar tabla Q legible
# --------------------------------------------------------
# Política: acción con mayor Q por estado. Estados inválidos (#) se marcan.
arrow_map = {0:'^', 1:'v', 2:'<', 3:'>'}

policy_chars = grid.copy()
for r in range(H):
    for c in range(W):
        if grid[r,c] in ['#','G','X','S']:
            continue
        s = to_state(r,c)
        a = np.argmax(Q[s])
        policy_chars[r,c] = arrow_map[a]

print("Política greedy aprendida (flechas) sobre el mapa:")
for r in range(H):
    print(''.join(policy_chars[r]))

# Tabla Q como DataFrame (solo para estados no pared)
rows = []
for r in range(H):
    for c in range(W):
        if is_wall(r,c):
            continue
        s = to_state(r,c)
        rows.append({
            "estado": f"({r},{c})",
            "Q_up(^)":   round(float(Q[s,0]), 2),
            "Q_down(v)": round(float(Q[s,1]), 2),
            "Q_left(<)": round(float(Q[s,2]), 2),
            "Q_right(>)":round(float(Q[s,3]), 2),
            "celda": grid[r,c]
        })

q_df = pd.DataFrame(rows)
display(q_df.head(10))

# --------------------------------------------------------
# 7) Demostración: ejecutar un episodio con la política aprendida
# --------------------------------------------------------
def greedy_action(state):
    qs = Q[state]
    max_q = np.max(qs)
    best_actions = np.flatnonzero(qs == max_q)
    return np.random.choice(best_actions)

s = reset()
trajectory = [from_state(s)]
total = 0
for t in range(50):
    a = greedy_action(s)
    s, r, done = step(s, a)
    total += r
    trajectory.append(from_state(s))
    if done:
        break

print("Trayectoria con política aprendida:", trajectory)
print("Recompensa total de la demostración:", total)

# Visualización de la trayectoria
vis = grid.copy()
for (rr,cc) in trajectory:
    if vis[rr,cc] == '.':
        vis[rr,cc] = '*'
print("Mapa con trayectoria (*):")
for r in range(H):
    print(''.join(vis[r]))

# --------------------------------------------------------
# 8) Guía de lectura de resultados
# --------------------------------------------------------
print("""
================= CÓMO LEER LOS RESULTADOS =================
1) Curva de recompensa:
   - Muestra sumas sin descuento de episodios con exploración ε-greedy.
   - Una tendencia ascendente es evidencia del entrenamiento, no prueba de optimalidad.
   - Evaluar la política greedy por separado distingue aprendizaje de exploración.

2) Tabla Q:
   - Q[s,a] estima el retorno esperado descontado de esa acción y su continuación.
   - Cerca de un pozo, entrar en él tiene valor bajo; otra acción puede conducir a la meta.
   - El valor depende de la acción y de la ruta, no solo de la cercanía a una casilla.

3) Política greedy aprendida (flechas):
   - Flechas muestran el argmax de la tabla Q estimada en cada celda libre.
   - La demostración comprueba una ruta desde S; no prueba optimalidad en todo el mapa.

4) Demostración:
   - 'Trayectoria' recorre celdas desde inicio hasta terminar.
   - Si termina en 'G' con pocos pasos y buena recompensa, la política es efectiva.

5) Parámetros:
   - α (learning rate) controla cuánto ajustas Q en cada paso.
   - γ (descuento) reduce el peso de recompensas futuras en Q; el gráfico suma sin descuento.
   - ε permite probar acciones alternativas; decae para aprovechar más lo aprendido.
   - Un entrenamiento finito no garantiza haber aprendido la mejor acción en todos los estados.
=============================================================
""")


### Después de ejecutar: tres salidas, tres preguntas

**1. Curva de episodios:** el eje horizontal cuenta intentos; el vertical suma los puntos recibidos en cada intento. Un punto bajo puede incluir pasos exploratorios o caída al pozo. No dibuja el camino físico ni es un porcentaje de aciertos.

**2. Mapa de flechas y tabla Q:** cada flecha señala una acción con la estimación más alta en esa casilla; si hay empate puede elegirse otra equivalente. La tabla muestra cuatro estimaciones por estado. `(0,0)` es fila 0, columna 0: los índices empiezan en cero; no son coordenadas geográficas. Filas con G o X son terminales y no requieren continuar moviéndose.

**3. Trayectoria final:** la lista de casillas y el mapa con * permiten seguir una demostración. Mira dónde termina y cuenta **movimientos = número de casillas de la lista − 1**, porque la primera casilla es el inicio. Si llega a G en cinco movimientos, recibió cuatro penalizaciones de −1 y +10 al llegar: total **6**. La ruta puede ser otra; comprueba la salida obtenida y no inventes que fue óptima en todo el mapa.

**Prueba pequeña:** en tu copia repite solo el fragmento «Demostración» desde `s = reset()` hasta su impresión, manteniendo Q fija. No repitas el entrenamiento. Elige la mejor estimación disponible sin exploración; puede desempatar rutas. Comprueba llegada, movimientos y puntos. Si modificas paredes o recompensas, debes volver a entrenar: las estimaciones anteriores correspondían a otro entorno.


### Lee, comprueba y explica

Distingue los intentos de entrenamiento de la ruta final. Una flecha representa una acción; la recompensa fue fijada por el ejercicio.

**Prueba pequeña:** Mover el número de episodios cambia cuánto practica el agente. No cambia por sí solo las recompensas ni convierte el tablero en una operación real.

**Vuelve al caso:** Registra estados, acciones y recompensas para evaluar un recorrido. Aprender una política requiere actualizarla a partir de la experiencia.

**Pregunta:** ¿Qué falta para que una ruta elegida manualmente sea aprendizaje por refuerzo?

**Respuesta razonada:** Que un agente use experiencias de estados, acciones y recompensas para actualizar su política o valores. Moverlo manualmente solo permite explorar el entorno y sus reglas.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/ml-refuerzo)


## Tema 3 · 1.6 · Comparar predicciones con respuestas reservadas

**Lo que vas a poder hacer:** Construir un prototipo de predicción con una decisión de uso documentada.

Antes de cambiar una operación, expliquemos la evidencia y qué falta comprobar.

### Entiende la idea

Cerrar una propuesta de Machine Learning significa poder contarla sin saltos: alguien necesita decidir, hay datos disponibles, una regla calcula una respuesta y comprobamos cuánto se equivoca. Una referencia es la solución sencilla que ya tenemos; una candidata es la propuesta que queremos probar. Las dos deben responder la misma pregunta sobre los mismos casos. La evidencia es lo que permite comprobar la mejora: datos separados, predicciones y errores. Un prototipo es una versión para probar esa idea, todavía no una solución autorizada para una operación real.

### Un caso pequeño antes del programa

**Una mejora pequeña con alcance concreto**

La cooperativa revisa una candidata para planear recogida. En tres fincas reservadas y ficticias ya conocemos el rendimiento en t/ha. La referencia predice siempre 4 t/ha y las predicciones candidatas están dadas. Las reglas se eligieron antes de mirar esta prueba. Con una superficie supuesta de 10 ha por finca, podemos traducir cada estimación a toneladas sin cambiar la comparación de rendimiento.

| Finca | Observado (t/ha) | Referencia constante (t/ha) | Candidata (t/ha) |
| --- | --- | --- | --- |
| A | 3,0 | 4,0 | 3,2 |
| B | 4,0 | 4,0 | 3,8 |
| C | 5,0 | 4,0 | 5,1 |

1. Referencia: |4−3| = 1, |4−4| = 0 y |4−5| = 1 t/ha. Su MAE es (1+0+1)/3 = 0,67 t/ha aproximadamente.
2. Candidata: |3,2−3| = 0,2; |3,8−4| = 0,2; |5,1−5| = 0,1. Su MAE es 0,5/3 = 0,17 t/ha aproximadamente.
3. En la finca A de 10 ha, la referencia estimaría 40 t y la candidata 32 t; se observaron 30 t. Esa comparación concreta ayuda a planear una prueba de recogida, pero no calcula costes ni decide capacidad de vehículos.

**Qué demuestra:** Podemos explicar esta mejora, sin afirmar una recomendación de fertilización ni beneficio económico real.

### Ahora identifica los datos de esta práctica

Elige una finca reservada y explica observado → predicción → diferencia. Después mira el error del conjunto, sin escoger solo el mejor caso.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** Los tres casos pequeños del aula explican la cuenta. Las 60 fincas del cuaderno aportan otra comparación: no copies las cifras de un conjunto al otro.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.cluster import DBSCAN
rng = np.random.default_rng(42)


In [ ]:
n = 240
X = pd.DataFrame({
    'lluvia_mm': rng.uniform(200, 800, n),
    'temperatura_c': rng.uniform(16, 30, n),
    'fertilizante_kg_ha': rng.uniform(50, 180, n)
})
y = (1 + .004 * X.lluvia_mm + .015 * X.fertilizante_kg_ha
     - .03 * (X.temperatura_c - 23)**2 + rng.normal(0, .25, n))
display(X.assign(rendimiento_t_ha=y).head())


### 1.2. Simular registros de lluvia que faltan

Ocultamos la lluvia de **20 fincas**, elegidas al azar sin repetir. `NaN` significa dato ausente; no significa cero lluvia. Mantenemos su rendimiento conocido: pudo registrarse la cosecha aunque faltara la medición de lluvia.

La respuesta se creó **antes** de ocultar los datos para poder evaluar este caso didáctico. Más adelante completaremos las entradas ausentes con una estimación llamada **imputación**.

**Lee la salida:** debe haber 20 valores de lluvia sin registro. Las filas mostradas ayudan a reconocer cómo aparece un dato ausente en una tabla.


In [ ]:
X.loc[rng.choice(n, 20, replace=False), 'lluvia_mm'] = np.nan
print('Fincas sin registro de lluvia:', X.lluvia_mm.isna().sum())
display(X.loc[X.lluvia_mm.isna()].head(3))


### 1.3. Separar antes de aprender

Reservamos **180 fincas para entrenamiento** y **60 para prueba**. `test_size=.25` aparta el 25 %; `random_state=42` reproduce esa partición.

`X_tr` y `y_tr` son entradas y respuestas de entrenamiento. `X_te` y `y_te` son las de prueba. El modelo aprende con las primeras; sus predicciones se comprueban con las respuestas reservadas.

La mediana para completar vacíos y la escala también se aprenden solo con entrenamiento. Usar prueba para calcularlas introduciría información que debía permanecer reservada: **fuga de información**.

Aquí tratamos las fincas como independientes. Para mediciones repetidas de una finca o pronósticos por fecha, habría que separar por finca o respetar el tiempo. En este ejemplo no elegimos hiperparámetros; si comparáramos muchas configuraciones, necesitaríamos validación dentro de entrenamiento.


In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=.25, random_state=42
)
print('Fincas de entrenamiento:', len(X_tr))
print('Fincas de prueba:', len(X_te))


### 1.4. Crear una referencia sencilla

Antes de usar una regresión, necesitamos una respuesta básica contra la cual comparar. `DummyRegressor(strategy='mean')` aprende la **media del rendimiento de entrenamiento** y predice ese mismo valor para todas las fincas de prueba. No utiliza lluvia, temperatura ni fertilizante para distinguirlas.

`make_pipeline` encadena pasos. En esta referencia, `SimpleImputer(strategy='median')` completa entradas ausentes y después actúa el predictor de media. La imputación mantiene una preparación compatible, aunque este predictor no depende de las entradas.

`fit` aprende esos pasos únicamente con entrenamiento. **Lee la salida:** el número es la predicción de referencia, en t/ha; no es su error.


In [ ]:
base = make_pipeline(
    SimpleImputer(strategy='median'),
    DummyRegressor(strategy='mean')
)
base.fit(X_tr, y_tr)
print(f'La referencia predice {base.predict(X_te)[0]:.3f} t/ha para cada finca.')


### 1.5. Aprender una regresión con las tres entradas

Este segundo pipeline hace tres pasos en orden:

1. **Imputar:** completa una lluvia ausente con la mediana aprendida de las lluvias de entrenamiento.
2. **Estandarizar:** expresa cada entrada respecto a su media y su dispersión, aprendidas con entrenamiento. Así se manejan magnitudes diferentes; esto no añade información ni garantiza una mejora.
3. **Regresión lineal:** aprende un valor inicial y un peso para cada entrada para estimar rendimiento.

Para datos de prueba, el pipeline aplica las mismas reglas y los pesos ya aprendidos; no los recalcula. La relación de temperatura que inventamos es curva y esta regresión usa las tres variables sin potencias: puede quedar error por esa simplificación y por la variación aleatoria.

**Lee la salida:** la mediana está en mm y es la que usará para cualquier lluvia ausente. No tiene por qué ser 600 mm: ese número pertenecía al ejemplo ilustrativo de cuatro fincas de la presentación.


In [ ]:
lineal = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LinearRegression()
)
lineal.fit(X_tr, y_tr)
print(f'Mediana de lluvia aprendida: {lineal[0].statistics_[0]:.3f} mm')


### 1.6. Comparar predicciones con respuestas reservadas

`predict(X_te)` estima rendimientos para las 60 fincas de prueba. Después comparamos cada predicción con su rendimiento conocido, `y_te`. Llamamos **error absoluto** a la distancia entre ambos valores, sin signo; el **MAE** es el promedio de esas distancias: `MAE = promedio(|y − ŷ|)`, donde `ŷ` es la predicción.

**Cómo leer la tabla y el gráfico:**

- Ambos modelos se evalúan sobre las mismas 60 fincas. Cada barra muestra su MAE, no el rendimiento promedio.
- El eje vertical usa t/ha. Un MAE de 0,46 significa un error absoluto promedio cercano a 0,46 toneladas por hectárea en esta prueba; no que todas las fincas tengan ese error.
- La barra más baja corresponde al menor error medio. Compara la regresión con la referencia para saber si las entradas ayudaron en este ejemplo.
- La última tabla muestra cinco fincas de prueba: respuesta conocida, predicción de regresión y error absoluto de cada una.

**Alcance:** con una ejecución completa desde la semilla 42, la referencia obtiene aproximadamente **0,705 t/ha** y la regresión **0,459 t/ha**. La regresión mejora esta referencia en esta simulación. No demuestra que funcionará igual en fincas reales, ni que una variable cause cambios en el rendimiento. Las cifras pueden cambiar si alteras los datos o ejecutas celdas aleatorias de nuevo sin reiniciar el recorrido.


In [ ]:
pred_base = base.predict(X_te)
pred_lineal = lineal.predict(X_te)
resultados = pd.DataFrame({
    'modelo': ['Media de entrenamiento', 'Regresión lineal'],
    'MAE_t_ha': [mean_absolute_error(y_te, pred_base),
                 mean_absolute_error(y_te, pred_lineal)]
})
display(resultados)
plt.bar(resultados.modelo, resultados.MAE_t_ha)
plt.ylabel('MAE en prueba (t/ha)')
plt.title('Comparar con una referencia: mismas 60 fincas de prueba')
plt.show()
comparacion = pd.DataFrame({
    'rendimiento_conocido_t_ha': y_te,
    'prediccion_lineal_t_ha': pred_lineal,
    'error_absoluto_t_ha': np.abs(y_te - pred_lineal)
})
display(comparacion.head().round(3))


### Lectura del gráfico 1 · ¿Las entradas mejoraron la referencia?

**Eje horizontal:** los dos modelos. **Eje vertical:** error absoluto medio en las 60 fincas de prueba, expresado en t/ha. La altura de una barra representa error, no producción ni número de fincas. Cada barra resume 60 comparaciones entre respuesta conocida y predicción.

En esta ejecución, la referencia tiene **0,705 t/ha** de MAE y la regresión **0,459 t/ha**. La segunda barra es más baja: la regresión comete menos error medio sobre la misma prueba. Eso es lo que podemos concluir para este ejemplo.

En la última tabla, una finca tiene rendimiento conocido **4,338**, predicción **4,889** y error absoluto **0,551 t/ha**. La predicción quedó por encima de la respuesta; otra fila puede quedar por debajo. El MAE promedia las distancias sin compensar unos errores con otros. No es un porcentaje de aciertos ni el error de todas las fincas.


**Conexión con la presentación:** la diapositiva 6 resume aprender y predecir; aquí ves las filas, los pasos y el error que produce ese recorrido. La diapositiva 8 usa cuatro fincas ilustrativas para explicar la mediana; este laboratorio calcula su propia mediana con sus 180 fincas de entrenamiento.

Documentación de consulta: [imputación](https://scikit-learn.org/stable/modules/impute.html), [pipelines](https://scikit-learn.org/stable/modules/compose.html#pipeline), [MAE](https://scikit-learn.org/stable/modules/model_evaluation.html#mean-absolute-error).


### Lee, comprueba y explica

Elige una finca reservada y explica observado → predicción → diferencia. Después mira el error del conjunto, sin escoger solo el mejor caso.

**Prueba pequeña:** Los tres casos pequeños del aula explican la cuenta. Las 60 fincas del cuaderno aportan otra comparación: no copies las cifras de un conjunto al otro.

**Vuelve al caso:** Reúne pregunta, datos, referencia, error y decisión de uso en un resultado que puedas volver a ejecutar.

**Pregunta:** ¿Qué evidencia necesitamos además de una métrica favorable?

**Respuesta razonada:** Casos representativos del uso esperado, separación correcta de datos, comparación justa con una referencia, revisión de errores y condiciones para supervisar o detener el uso. Tres casos ilustrativos no bastan.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/ml-integracion)


## Cierra el encuentro con una explicación

1. **El futuro va después:** Registra fecha de corte, horizonte y error. Compara las reglas sobre las mismas fechas posteriores al corte.
2. **Aprender de las consecuencias:** Registra estados, acciones y recompensas para evaluar un recorrido. Aprender una política requiere actualizarla a partir de la experiencia.
3. **Explicar el resultado antes de usarlo:** Reúne pregunta, datos, referencia, error y decisión de uso en un resultado que puedas volver a ejecutar.

En cada resultado identifica entrada, salida, unidad o significado, forma de comprobarlo y límite. Guarda tu copia personal. Colab no registra automáticamente tu avance en el aula.
